# 01 — Aquisição e inspeção

Valida a fonte local, extrai o ZIP de forma idempotente, inventaria os 32 arquivos e cria uma amostra de no máximo 1%/5.000 linhas. Não realiza junção entre estudantes.

In [1]:
from pathlib import Path
import sys

# O notebook pode ser executado a partir da raiz ou da pasta notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /Users/djalma.rodrigues/projetos/99_Revisar/projeto-ia-ciencia-dados-2026


## Requisitos e fonte

Os dois PDFs obrigatórios do professor foram lidos antes da implementação; a matriz técnica está em `docs/requisitos_extraidos_dos_pdfs.md`. O dataset é o ENADE 2023 do INEP, tabular, em TXT delimitado por `;`. O pacote não traz licença convencional; os termos e a LGPD constam no manual oficial.

In [2]:
from zipfile import ZipFile
from src.config import RAW_DATA_DIR

zip_path = ROOT / "microdados_enade_2023.zip"
expected = RAW_DATA_DIR / "microdados2023_arq1.txt"
assert zip_path.exists(), "ZIP de origem não encontrado na raiz do projeto."
if not expected.exists():
    # Extração restrita ao diretório data/raw deste projeto.
    with ZipFile(zip_path) as archive:
        archive.extractall(ROOT / "data" / "raw")
print("Dados extraídos:", expected.exists())

Dados extraídos: True


In [3]:
import hashlib
import pandas as pd

# Inventário leve: conta linhas por streaming e lê somente o cabeçalho.
rows = []
for path in sorted(RAW_DATA_DIR.glob("microdados2023_arq*.txt"), key=lambda p: int(p.stem.split("arq")[-1])):
    with path.open("r", encoding="utf-8-sig", errors="replace") as handle:
        n_rows = sum(1 for _ in handle) - 1
    columns = pd.read_csv(path, sep=";", nrows=0).columns.tolist()
    rows.append({"arquivo": path.name, "linhas": n_rows, "colunas": len(columns), "bytes": path.stat().st_size})
inventory = pd.DataFrame(rows)
inventory.to_csv(ROOT / "reports" / "tables" / "inventario_arquivos.csv", index=False, encoding="utf-8-sig")
display(inventory)
print("Total descompactado (MB):", round(inventory["bytes"].sum() / 1024**2, 2))

sha256 = hashlib.sha256()
with zip_path.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        sha256.update(chunk)
print("SHA-256 do ZIP:", sha256.hexdigest())

,arquivo,linhas,colunas,bytes
0,microdados2023_arq1.txt,406294,10,17834657
1,microdados2023_arq2.txt,406294,5,10178112
2,microdados2023_arq3.txt,406294,44,92214519
3,microdados2023_arq4.txt,406294,44,37810512
4,microdados2023_arq5.txt,406294,3,6115208
5,microdados2023_arq6.txt,406294,3,6521503
6,microdados2023_arq7.txt,406294,3,6076641
7,microdados2023_arq8.txt,406294,3,6076639
8,microdados2023_arq9.txt,406294,3,6076633
9,microdados2023_arq10.txt,406294,3,6076638


Total descompactado (MB): 313.79
SHA-256 do ZIP: 7565eb919b7403e2e95c61bd26ff9d3a90311a6897087ec5c237d53693745853


## Amostra e inspeção do arquivo de resultados

In [4]:
from src.data import read_raw
from src.config import RANDOM_STATE, INTERIM_DIR

results = read_raw(3)
sample_size = min(5_000, int(len(results) * 0.01))
sample = results.sample(n=sample_size, random_state=RANDOM_STATE).sort_index()
sample.to_csv(INTERIM_DIR / "amostra_arq3_1pct.csv", index=False, encoding="utf-8-sig")
print(f"Base: {results.shape}; amostra: {sample.shape} ({sample_size / len(results):.3%})")
display(sample.head())
display(results[["TP_PRES", "NT_GER"]].describe(include="all"))
display(results["TP_PRES"].value_counts(dropna=False).rename("quantidade"))

Base: (406294, 44); amostra: (4062, 44) (1.000%)


,NU_ANO,CO_CURSO,NU_ITEM_OFG,NU_ITEM_OFG_Z,NU_ITEM_OFG_X,NU_ITEM_OFG_N,NU_ITEM_OCE,NU_ITEM_OCE_Z,NU_ITEM_OCE_X,NU_ITEM_OCE_N,...,NT_CE_D1,CO_RS_I1,CO_RS_I2,CO_RS_I3,CO_RS_I4,CO_RS_I5,CO_RS_I6,CO_RS_I7,CO_RS_I8,CO_RS_I9
302,2023,1313394,9,0,0,0,29,0,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
305,2023,1401591,9,0,0,0,29,0,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
333,2023,1401591,9,0,0,0,29,0,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
416,2023,1279375,9,0,0,0,29,0,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
419,2023,1458671,9,0,0,0,29,0,6,0,...,NaN,NaN,NaN,NaN,B,NaN,NaN,B,NaN,NaN


,TP_PRES,NT_GER
count,406294.000000,346557.000000
mean,508.127861,48.284498
std,115.686258,15.380657
min,222.000000,0.000000
25%,555.000000,36.800000
50%,555.000000,47.900000
75%,555.000000,59.400000
max,888.000000,99.000000


TP_PRES
555    346728
222     56734
565      1251
444       600
334       551
585       392
888        38
Name: quantidade, dtype: int64

## Restrição LGPD

O manual afirma que arquivos diferentes foram ordenados por variáveis distintas. A linha *i* de um arquivo não corresponde à linha *i* de outro. Logo, as análises seguintes agregam cada arquivo por `CO_CURSO` antes de qualquer `merge`.